# 03 - Risk index and map
Combines exposure and vulnerability into a risk index, labels tracts by quadrant, makes a static and interactive map, and optionally publishes to ArcGIS.

In [ ]:
import sys; sys.path.insert(0, "..")
import geopandas as gpd, pandas as pd, matplotlib.pyplot as plt
from src import config
from src.scoring import compute_risk_index, label_quadrant

tracts = gpd.read_file(config.PROCESSED / "tracts_exposure.geojson")
scored = compute_risk_index(tracts, config.EXPOSURE_WEIGHTS, config.RISK_WEIGHTS, vulnerability_col="svi")
scored["quadrant"] = label_quadrant(scored, config.HIGH_THRESHOLD)
scored["quadrant"].value_counts(dropna=False)

## Priority tracts (high exposure and high vulnerability)

In [ ]:
top = (scored[scored["quadrant"] == "High exposure / High vulnerability"]
       .sort_values("risk", ascending=False)
       [["GEOID", "exposure", "vulnerability", "risk", "sfha_share", "below_2ft", "claims_rate"]])
top.head(15)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (col, title, cmap) in zip(axes, [("exposure","Exposure","Blues"),("vulnerability","Vulnerability (SVI)","Oranges"),("risk","Combined risk","Reds")]):
    scored.plot(column=col, cmap=cmap, legend=True, ax=ax, linewidth=0, missing_kwds={"color":"lightgrey"})
    ax.set_title(title); ax.axis("off")
plt.tight_layout()
plt.savefig(config.OUTPUTS / "risk_map.png", dpi=200)

## Sensitivity check
Do the top tracts change much if the weights change? Stable results are more credible.

In [ ]:
alt_weights = {"exposure": 0.4, "vulnerability": 0.6}
alt = compute_risk_index(tracts, config.EXPOSURE_WEIGHTS, alt_weights, vulnerability_col="svi")
n = 50
a = set(scored.nlargest(n, "risk")["GEOID"]); b = set(alt.nlargest(n, "risk")["GEOID"])
print(f"Top-{n} overlap between weighting schemes: {len(a & b)/n:.0%}")

## Interactive map and export

In [ ]:
scored[["GEOID","exposure","vulnerability","risk","quadrant","sfha_share","below_1ft","below_2ft","below_3ft","claims_rate","geometry"]] \
    .to_file(config.OUTPUTS / "tract_risk.geojson", driver="GeoJSON")
m = scored.explore(column="risk", cmap="Reds", tooltip=["GEOID","risk","exposure","vulnerability","quadrant"], tiles="CartoDB positron")
m.save(config.OUTPUTS / "risk_map.html")
m

## (Optional) Publish to ArcGIS
Requires a free ArcGIS account and credentials in `.env`. Skip this cell if you don't have one.

In [ ]:
import os
from dotenv import load_dotenv
from arcgis.gis import GIS

load_dotenv(config.ROOT / ".env")
gis = GIS(os.getenv("ARCGIS_PORTAL_URL", "https://www.arcgis.com"),
          os.getenv("ARCGIS_USERNAME"), os.getenv("ARCGIS_PASSWORD"))
print("Signed in as", gis.users.me.username)

item = gis.content.add(
    {"title": "South Florida Flood Risk by Tract", "type": "GeoJson",
     "tags": ["flood", "south florida", "sea level rise"]},
    data=str(config.OUTPUTS / "tract_risk.geojson"),
)
published = item.publish()          # hosted feature layer (uses little to no credits for small data)
print(published.homepage)